<a href="https://colab.research.google.com/github/Alexandraqq1/RetuRO-Data-Analysis/blob/main/RetuRO_ETL_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pdfplumber
import pandas as pd

# Calea catre directorul de lucru
folder = "/content/"
toate_datele = []

# Separam coloanele pe baza spațiilor albe dintre cuvinte
setari = {"vertical_strategy": "text", "horizontal_strategy": "text"}

for fisier in os.listdir(folder):
    # Filtrăm să deschidă doar PDF-uri
    if fisier.endswith(".pdf"):
        cale = os.path.join(folder, fisier)

        # Extrag anul direct din numele fisierului pentru a ști ce format are tabelul
        anul = "Necunoscut"
        if "24" in fisier: anul = "2024"
        elif "25" in fisier: anul = "2025"
        elif "26" in fisier: anul = "2026"

        with pdfplumber.open(cale) as pdf:
            for pagina in pdf.pages:
                tabel = pagina.extract_table(setari)
                if not tabel: continue

                df = pd.DataFrame(tabel)
                if len(df.columns) < 8: continue

                df = df.iloc[:, :8]
                # Atribuim nume temporare scurte pentru o manipulare mai ușoară
                df.columns = ["C0", "C1", "PB", "PK", "MB", "MK", "SB", "SK"]

                # Curatam virgulele și spațiile din numere
                for col in ["PB", "PK", "MB", "MK", "SB", "SK"]:
                    df[col] = df[col].astype(str).str.replace(r'[, ]', '', regex=True)

                # Pastram doar rândurile care au un număr la coloana Plastic_Buc
                df = df[df['PB'].str.isnumeric()]

                if anul == "2024":
                    df['Luna'] = df['C0']
                    df['Judet'] = df['C1']
                else:
                    df['Luna'] = fisier.replace(".pdf", "") # Iau luna din titlul fișierului
                    df['Judet'] = df['C0']

                df['Anul'] = anul

                # Reordonare coloanele în formatul final
                df_final = df[['Anul', 'Luna', 'Judet', 'PB', 'PK', 'MB', 'MK', 'SB', 'SK']].copy()
                df_final.columns = ["Anul", "Luna", "Judet", "Plastic_Buc", "Plastic_Kg", "Metal_Buc", "Metal_Kg", "Sticla_Buc", "Sticla_Kg"]
                toate_datele.append(df_final)

if toate_datele:
    tabel_final = pd.concat(toate_datele, ignore_index=True)


    def curata_luna(text):
        text = str(text).lower()
        if 'ian' in text: return 'Ianuarie'
        if 'feb' in text: return 'Februarie'
        if 'mar' in text: return 'Martie'
        if 'apr' in text: return 'Aprilie'
        if 'mai' in text: return 'Mai'
        if 'iun' in text: return 'Iunie'
        if 'iul' in text: return 'Iulie'
        if 'aug' in text: return 'August'
        if 'sep' in text: return 'Septembrie'
        if 'oct' in text: return 'Octombrie'
        if 'nov' in text or 'noi' in text: return 'Noiembrie'
        if 'dec' in text: return 'Decembrie'
        return text

    tabel_final['Luna'] = tabel_final['Luna'].apply(curata_luna)


    #Salvam tot intr un excel
    tabel_final.to_excel("/content/Date_RetuRO_Curatate.xlsx", index=False)
    print("Fisier Excel creat și lunile au fost standardizate!")
    display(tabel_final.head())
else:
    print("Eroare: Nu se pot extrage datele.")